<a href="https://colab.research.google.com/github/astronomy-commons/lsdb-foundation-model/blob/main/notebooks/tokenize_legacysurvey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tokenizing Legacy Survey images with AION-1

This notebook tokenizes galaxy cutouts from the
[`UniverseTBD/mmu_ssl_legacysurvey_north`](https://huggingface.co/datasets/UniverseTBD/mmu_ssl_legacysurvey_north) Hugging Face dataset
(the northern Legacy Surveys DR9 sample of the Multimodal Universe: 14M objects with
g, r, z cutouts, stored as a HATS catalog) with the
[AION-1](https://github.com/PolymathicAI/AION) image codec, using the `aion-hats` library
from [`astronomy-commons/lsdb-foundation-model`](https://github.com/astronomy-commons/lsdb-foundation-model).

The output is again a HATS catalog with the **same columns as the original catalog, minus
the images, plus a `tok_image` column** of 576 discrete tokens per object, stored as a
nested column (`struct<token: list<int64>>`) that `lsdb` understands natively. The very same
function scales to the whole catalog on a multi-GPU cluster (see the last section).

For the demo we only process 100 objects and do not push anything to the Hub.

### Enabling GPU access

The codec runs on CPU too, but on Colab go to `Runtime > Change runtime type` and select a
GPU to speed things up.

### Installing dependencies

In [ ]:
!pip install --quiet git+https://github.com/astronomy-commons/lsdb-foundation-model.git lsdb matplotlib

## Step I: open the catalog

`open_catalog` reads the HATS metadata (`hats.properties`, `partition_info.csv`, the
parquet schema) without downloading any data. The catalog is made of ~11,000 HEALPix
partitions of up to 8192 objects each; those partitions are the unit of work of the
tokenizer. `detect_modalities` lists the columns AION has a codec for: the `image` struct
(bands + flux) maps to `LegacySurveyImage`, and scalar columns are matched to AION's
scalar modalities by name (`flux_g`, `ebv`, ...).

In [ ]:
from aion_hats import open_catalog, detect_modalities

SOURCE = "UniverseTBD/mmu_ssl_legacysurvey_north"

catalog = open_catalog(SOURCE)
print(catalog)
print(f"{catalog.properties['hats_nrows']} rows, orders {sorted({p.order for p in catalog.partitions})}")
print(catalog.schema)

for spec in detect_modalities(catalog.schema, catalog_name=catalog.name, sample=lambda: catalog.sample(2)):
    print(spec)

## Step II: tokenize

`tokenize_catalog` streams the rows of each partition in batches, wraps the cutouts in
AION's `LegacySurveyImage` modality (the codec knows the survey and the bands, and handles
the missing `i` band), encodes them into 576 tokens (a 24x24 grid over the central 96x96
pixels) and writes the tokenized partition next to the untouched columns.

Every column AION has a codec for is tokenized: the image into `tok_image`, and the scalars
(`flux_g`, `ebv`, `z_spec`, `ra`, ...) into `tok_flux_g`, `tok_ebv`, `tok_z`, `tok_ra`, ...;
pass `modalities=["image"]` to restrict the run. We stop after 100 objects, in which case
only the needed row groups of the first partition are streamed from the Hub.

In [ ]:
from aion_hats import tokenize_catalog

OUTPUT = "tokenized_demo"

summary = tokenize_catalog(SOURCE, OUTPUT, max_rows=100, batch_size=32)
print(summary)

## Step III: look at the result

The output directory is a HATS catalog: `hats.properties`, `partition_info.csv` and a
`dataset/` tree of parquet partitions. `aion_hats.json` records the provenance (source,
modalities, codec and library versions).

In [ ]:
import pyarrow.dataset as ds

!find $OUTPUT -type f | sort

tokens = ds.dataset(f"{OUTPUT}/dataset", format="parquet", exclude_invalid_files=True).to_table()
print(tokens.schema)
row = tokens.slice(0, 1).to_pylist()[0]
print({k: v for k, v in row.items() if k != "tok_image"})
print("tokens:", len(row["tok_image"]["token"]), row["tok_image"]["token"][:16], "...")

The parquet files also load as a regular Hugging Face dataset (the generated `README.md`
carries the matching `data_files` config for when the folder is uploaded to the Hub):

In [ ]:
from datasets import load_dataset

hf_dataset = load_dataset("parquet", data_files=f"{OUTPUT}/dataset/**/*.parquet", split="train")
print(hf_dataset)

Because the layout is preserved, `lsdb` opens the tokenized catalog like any other HATS
catalog (and could cross-match or join it with the source on `_healpix_29`). The token
column is recognised as a *nested* column: each object carries a small sub-table with a
`token` column, and `tokenized["tok_image.token"]` flattens it.

In [ ]:
import lsdb

tokenized = lsdb.open_catalog(OUTPUT)
print(tokenized.dtypes)
df = tokenized.compute()
print(df["tok_image"].iloc[0].head())
df.head(3)

## Step IV: decode the tokens

The same codec decodes tokens back into a 96x96 cutout, a useful sanity check of what
information the tokenization retains. We read the original images of the first four
objects directly from the source partition for comparison.

In [ ]:
import numpy as np
import pyarrow.compute as pc
import torch
import matplotlib.pyplot as plt
from aion.codecs import CodecManager
from aion.modalities import LegacySurveyImage
from aion_hats import default_device

device = default_device()
codec_manager = CodecManager(device=device)

originals = catalog.read_partition(catalog.partitions[0], columns=["object_id", "image"], max_rows=4)
token_batch = torch.as_tensor(np.stack(pc.struct_field(tokens.slice(0, 4).column("tok_image"), "token").to_pylist()), device=device)
decoded = codec_manager.decode({LegacySurveyImage.token_key: token_batch}, LegacySurveyImage, bands=["DES-G", "DES-R", "DES-Z"])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k in range(4):
    flux = np.asarray(originals.column("image")[k]["flux"].as_py())
    axes[0, k].imshow(flux[1, 28:-28, 28:-28], cmap="gray")
    axes[0, k].set_title(f"{originals.column('object_id')[k]} (r, input)")
    axes[1, k].imshow(decoded.flux[k, 1].cpu().numpy(), cmap="gray")
    axes[1, k].set_title("decoded from tokens")
for ax in axes.ravel():
    ax.axis("off")

## Step V: train a small model on the tokens

`aion_hats.train` trains the AION transformer (the `FM` model of the `aion` package) on a tokenized
catalog read through `lsdb`. Here a tiny model for a handful of steps on the 100-object demo, just to
see the loop run; `configs/ls_north_base.yaml` in the repository is the real recipe (one process per
GPU with `srun`, FSDP, bf16). Everything in the config can be overridden with `key.path=value` strings.

In [ ]:
import torch
from aion_hats.train import load_config, train

cfg = load_config(
    None,
    [
        f"data.datasets=[{{name: demo, catalog: {OUTPUT}, modalities: [tok_image]}}]",
        "model.preset=tiny",
        "data.num_workers=2",
        "run.batch_size=16",
        "run.output_dir=train_demo",
        f"run.dtype={'bfloat16' if torch.cuda.is_available() else 'float32'}",
        "schedule.epochs=2",
        "schedule.steps_per_epoch=5",
        "schedule.warmup_steps=2",
        "run.print_freq=1",
    ],
)
train(cfg)

The run directory holds `checkpoint-<epoch>.pth` (re-running resumes from the last one), a `log.txt`
with one JSON line per epoch, and `final/`, the model in the Hugging Face layout that the rest of the
AION toolchain loads:

In [ ]:
from aion import AION

model = AION.from_pretrained("train_demo/final")
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters, modalities: {list(model.encoder_embeddings)}")

## Scaling up

The full catalog is 14M objects in 3.4 TiB of parquet. The same `tokenize_catalog` call
(or the `aion-hats tokenize` command) does the whole job: without `max_rows`, each worker
downloads its partitions in the background, tokenizes them, writes each one atomically and
skips partitions that already exist, so a job resumes by re-running it. Workers take their
rank and GPU from the environment (`SLURM_PROCID`, `RANK`, ...), so on a SLURM cluster such
as Perlmutter one process per GPU is:

```bash
#SBATCH -C gpu -N 4 --ntasks-per-node=4 --gpus-per-task=1
srun aion-hats tokenize UniverseTBD/mmu_ssl_legacysurvey_north $SCRATCH/ls_north_tokens \
    --batch-size 256 --cache-dir $SCRATCH/stage --num-prefetch 2
aion-hats finalize $SCRATCH/ls_north_tokens   # once, after all workers are done
```

On a single multi-GPU machine, `--num-procs 4` spawns one worker per GPU. The finalized
folder is ready for `huggingface_hub.HfApi().upload_folder(...)` (not done here).